### PsAID Mixture Model Clustering

FMM models have been computed in STATA.

See DO FILE: Documents/Python/PsAID-Cluster-Analysis/PsAID_Mixture Models/FMMs V2 Oct 2 2025.do

## Initialization

In [586]:
## Set up
## Import Packages
import warnings
import os

import pandas as pd
import numpy as np
import re # Regular Expressions

import scipy
from scipy.stats import norm, skew, kurtosis, spearmanr, iqr
from scipy.spatial.distance import cdist
from scipy.optimize import linear_sum_assignment

import sklearn
from sklearn.metrics import silhouette_score, silhouette_samples, adjusted_rand_score, fowlkes_mallows_score, davies_bouldin_score, calinski_harabasz_score
from sklearn.manifold import TSNE

import matplotlib.pyplot as plt
import seaborn as sns
import umap

# Suppress Future Warnings
warnings.simplefilter(action='ignore', category=FutureWarning)
import contextlib #Suppress prints during when executing functions
import io

In [587]:
## Import Dataset
# PsAID Dataset
data_complete = pd.read_csv('/Users/pattydegroot/Documents/Python/PsAID-Cluster-Analysis/PsAID_Pre-Processing & Inspection/PsAID_Preprocessed.csv') # Import dataset
data_complete.head() # Check data
# Drop unnessecary variables
columns_gp = [i for i in data_complete.columns if re.match(r'^gp\d{2}$',i)] #Select all columns starting with 'gp' and ending after the 2 digits.
psaid12 = data_complete[['respondentid', 'mm', 'submitdate'] + columns_gp + ['psaid12']]

### Choose Clustering Solution

In [588]:
algorithm = 'GSEM'
#algorithm = 'FMM'

compute_tsne = False
compute_umap = False
compute_silhouette = False
compute_pairplot = False

### Import Dataset

In [589]:
# Import Mixture Clusters
Mixture_data = pd.read_stata(f'/Users/pattydegroot/Documents/Python/PsAID-Cluster-Analysis/PsAID_Mixture Models/PsAID_{algorithm}_Clustering.dta') # Import dataset

# Rename cluster vars for compatibility
def format_variable(name):
    name = re.sub(r'^cluster_', 'clusterID_', name, flags=re.IGNORECASE) # Replace "Cluster_" (case-insensitive) with "clusterID_
    name = re.sub(r'_(K\d+)$', lambda m: '_' + m.group(1).lower(), name) # Lowercase the final part if it matches 'K' followed by digits
    return name
    
cluster_columns = [i for i in Mixture_data.columns if re.match(r'^Cluster',i)]
rename_map = {col: format_variable(col) for col in cluster_columns} # Build the rename mapping
Mixture_data.rename(columns=rename_map, inplace=True) # Rename in the DataFrame
cluster_columns = [i for i in Mixture_data.columns if re.match(r'^clusterID_',i)]

# Format MM
# Map categories
mm_categoriesA = ['Intake ronde', 'T3', 'T6', 'T9', 'T12', 'T18', 'T24', 'T36', 'T48', 'T60', 'T72', 'T84', 'T96', 'T108' ] # List of original categories
mm_categoriesB = ['0', '3', '6', '9', '12', '18', '24', '36', '48', '60', '72', '84', '96', '108'] # List of new categories
Mixture_data['mm'] = Mixture_data['mm'].replace(dict(zip(mm_categoriesA ,mm_categoriesB))) #replace original with new categories

# Convert objects to integers
Mixture_data[['mm']+cluster_columns] = Mixture_data[['mm'] + cluster_columns].astype('int')
assert Mixture_data['mm'].dtype == 'int' #Check if the mm are indeed integers

# Check if converted
unique_values_mm = sorted(Mixture_data['mm'].unique())
print(unique_values_mm) #check if replaced
Mixture_data['mm'].info()

# Subset the FMM data
Mixture_data = Mixture_data[['respondentid', 'mm'] + cluster_columns] 
Mixture_data.head() # Check data

[0, 3, 6, 9, 12, 18, 24, 36, 48, 60, 72, 84, 96, 108]
<class 'pandas.core.series.Series'>
RangeIndex: 5099 entries, 0 to 5098
Series name: mm
Non-Null Count  Dtype
--------------  -----
5099 non-null   int64
dtypes: int64(1)
memory usage: 40.0 KB


,respondentid,mm,clusterID_gsemGauss_k3,clusterID_gsemGauss_k4,clusterID_gsemGauss_k5,clusterID_gsemGauss_k6,clusterID_gsemGauss_k7,clusterID_gsemPoisson_k3,clusterID_gsemPoisson_k4,clusterID_gsemPoisson_k5,clusterID_gsemPoisson_k6,clusterID_gsemPoisson_k7,clusterID_gsemWeibull_k3,clusterID_gsemWeibull_k4,clusterID_gsemWeibull_k5,clusterID_gsemWeibull_k6,clusterID_gsemWeibull_k7
0,10173314.0,0,1,4,4,3,4,3,3,1,2,2,1,3,3,5,1
1,10173314.0,3,1,4,4,3,2,3,3,1,2,2,3,3,3,5,1
2,10173314.0,6,3,1,3,4,3,2,1,5,1,4,1,4,2,6,7
3,10173314.0,9,3,1,3,4,3,2,1,5,1,4,1,4,2,6,7
4,10173314.0,12,3,1,3,4,3,2,4,4,6,4,2,1,2,6,7


In [590]:
# Merge FMM data with data_complete
print(f'Shape_prior: {data_complete.shape}')
data_complete = pd.merge(data_complete, Mixture_data, on = ['respondentid', 'mm'], how = 'left')
print(f'Shape_after: {data_complete.shape}')
data_complete.columns

Shape_prior: (5099, 37)
Shape_after: (5099, 52)


Index(['respondentid', 'mm', 'submitdate', 'gp01', 'gp02', 'gp03', 'gp04',
       'gp05', 'gp06', 'gp07', 'gp08', 'gp09', 'gp10', 'gp11', 'gp12',
       'psaid12', 'Geslacht', 'age', 'bmi', 'packyear', 'pm86b', 'popsjc',
       'poptjc', 'lei_mases', 'dactcount', 'bsa', 'crp', 'fv101a', 'fv103a',
       'fv102a', 'dapsa', 'dapsa_state', 'pasdas', 'pasdas_state', 'mda_state',
       'psaid_state', 'pass_state', 'clusterID_gsemGauss_k3',
       'clusterID_gsemGauss_k4', 'clusterID_gsemGauss_k5',
       'clusterID_gsemGauss_k6', 'clusterID_gsemGauss_k7',
       'clusterID_gsemPoisson_k3', 'clusterID_gsemPoisson_k4',
       'clusterID_gsemPoisson_k5', 'clusterID_gsemPoisson_k6',
       'clusterID_gsemPoisson_k7', 'clusterID_gsemWeibull_k3',
       'clusterID_gsemWeibull_k4', 'clusterID_gsemWeibull_k5',
       'clusterID_gsemWeibull_k6', 'clusterID_gsemWeibull_k7'],
      dtype='object')

In [591]:
data_complete.head()

,respondentid,mm,submitdate,gp01,gp02,gp03,gp04,gp05,gp06,gp07,...,clusterID_gsemPoisson_k3,clusterID_gsemPoisson_k4,clusterID_gsemPoisson_k5,clusterID_gsemPoisson_k6,clusterID_gsemPoisson_k7,clusterID_gsemWeibull_k3,clusterID_gsemWeibull_k4,clusterID_gsemWeibull_k5,clusterID_gsemWeibull_k6,clusterID_gsemWeibull_k7
0,78195329,36,2019-03-31 22:06:01,2,2,2,2,2,0,1,...,2,1,5,1,4,1,4,2,6,7
1,10862915,6,2019-02-06 14:26:32,1,6,2,1,1,1,1,...,3,1,5,1,2,1,4,5,5,5
2,29370588,18,2016-01-06 13:28:33,6,7,5,6,4,6,7,...,1,2,2,4,1,3,2,4,2,6
3,87648336,18,2016-08-18 00:05:35,0,4,5,0,0,0,0,...,2,4,5,1,4,2,4,2,6,7
4,42677901,84,2022-04-25 21:30:01,0,0,0,0,0,0,0,...,2,4,4,6,3,2,1,1,1,2


In [592]:
## Dictionary
domains_dict = {
    'gp01': 'Pain',
    'gp02': 'Fatigue',
    'gp03': 'Skin',
    'gp04': 'Work-recreational activities',
    'gp05': 'Bodily functioning',
    'gp06': 'Discomfort',
    'gp07': 'Sleep',
    'gp08': 'Coping',
    'gp09': 'Anxiety',
    'gp10': 'Shame',
    'gp11': 'Social activities',
    'gp12': 'Depression'
}

## Plot Internal Cluster Quality Scores

In [593]:
# Compute Scores
results = []

for cluster in cluster_columns:
    labels = data_complete[cluster]

    values, counts = np.unique(labels, return_counts=True) #Compute cluster sizes

    # Compute Quality Scores
    s_score_OG = silhouette_score(psaid12[columns_gp], labels)
    ch_index_OG = calinski_harabasz_score(psaid12[columns_gp], labels)
    db_index_OG = davies_bouldin_score(psaid12[columns_gp], labels)

    # Store Quality Metricx
    results.append({
        'cluster_name': cluster,
        'silhouette_score': s_score_OG,    # Computed on Input data. Higher values indicate better separation
        'calinski_harabasz': ch_index_OG,  # Computed on Input data. Higher values indicate better separation. Within-cluster compactness vs the between-cluster dispersion.
        'davies_bouldin': db_index_OG,     # Computed on Input data. Lower values indicate clusters are distinct. Measures average similarity between each cluster and its most similar other cluster
        'cluster sizes': counts,              # Datapoints per cluster
    })

In [594]:
pd.DataFrame(results)

,cluster_name,silhouette_score,calinski_harabasz,davies_bouldin,cluster sizes
0,clusterID_gsemGauss_k3,0.370044,4159.593940,1.260955,"[1467, 952, 2680]"
1,clusterID_gsemGauss_k4,0.310949,3267.182330,1.538487,"[2377, 949, 498, 1275]"
2,clusterID_gsemGauss_k5,0.280381,2668.888257,1.785943,"[313, 659, 2240, 1161, 726]"
3,clusterID_gsemGauss_k6,0.269714,2286.534401,1.821382,"[580, 357, 1110, 2143, 392, 517]"
4,clusterID_gsemGauss_k7,0.253388,1977.873447,2.030382,"[454, 353, 2092, 1031, 339, 561, 269]"
5,clusterID_gsemPoisson_k3,0.301692,3786.571609,1.212774,"[1375, 1907, 1817]"
6,clusterID_gsemPoisson_k4,0.205570,3017.163119,1.473069,"[1447, 979, 1346, 1327]"
7,clusterID_gsemPoisson_k5,0.193608,2442.178777,1.839092,"[679, 1003, 746, 1271, 1400]"
8,clusterID_gsemPoisson_k6,0.154917,2142.186457,1.941475,"[1169, 591, 690, 699, 882, 1068]"
9,clusterID_gsemPoisson_k7,0.125358,1891.768218,2.190215,"[811, 632, 926, 1058, 543, 630, 499]"


### tSNE

In [595]:
## Compute t-SNE 2D
def tSNE_2D(data, cluster_columns, columns_gp, RandomSeed, figure_storage):
    tsne2D = pd.DataFrame(TSNE(n_components=2, random_state=RandomSeed).fit_transform(data[columns_gp]), columns=['tSNE1', 'tSNE2'])

    for describe_clusters in cluster_columns:
        ## Add clusterIDs
        tsne2D[describe_clusters] = data[describe_clusters]
        
        ## Plot t-SNE Visualisation
        plt.figure(figsize=(8, 6))
        plt.scatter(tsne2D['tSNE1'], tsne2D['tSNE2'], c=tsne2D[describe_clusters], cmap='viridis', alpha=0.7)
        plt.xlabel('t-SNE 1')
        plt.ylabel('t-SNE 2')
        plt.title(f'2D t-SNE Visualization of {describe_clusters}')
        #ax.legend([0, 1, 2, 3])
        plt.show()

In [596]:
if compute_tsne:
    tSNE_2D(data_complete, cluster_columns, columns_gp, RandomSeed=42, figure_storage=None)

### UMAP

Uniform Manifold Approximation and Projection (UMAP) is a dimension reduction technique that can be used for visualisation similarly to t-SNE, but also for general non-linear dimension reduction. 

[ref] McInnes, L, Healy, J, UMAP: Uniform Manifold Approximation and Projection for Dimension Reduction, ArXiv e-prints 1802.03426, 2018 <br>
[ref] Healy, J., McInnes, L. Uniform manifold approximation and projection. Nat Rev Methods Primers 4, 82 (2024). <br>

[implementation] https://pypi.org/project/umap-learn/ 

In [597]:
def UMAP_2D(data, cluster_columns, columns_gp, RandomSeed, figure_storage):
    UMAP = pd.DataFrame(umap.UMAP(random_state=RandomSeed, verbose=False).fit_transform(data[columns_gp]), columns=['UMAP1', 'UMAP2'])
    
    for describe_clusters in cluster_columns:
        ## Add clusterIDs
        UMAP[describe_clusters] = data[describe_clusters]
        
        ## Plot t-SNE Visualisation
        plt.figure(figsize=(8, 6))
        plt.scatter(UMAP['UMAP1'], UMAP['UMAP2'], c=UMAP[describe_clusters], cmap='viridis', alpha=0.7)
        plt.xlabel('UMAP 1')
        plt.ylabel('UMAP 2')
        plt.title(f'UMAP Visualization of {describe_clusters}')
        #ax.legend([0, 1, 2, 3])
        plt.show()

In [598]:
if compute_umap:
    UMAP_2D(data_complete, cluster_columns, columns_gp, RandomSeed=42, figure_storage=None)

### Silhouette Plots

In [599]:
def compute_silhouette_plot(data, columns_gp, cluster, figure_storage):
    """
    Function that calculates and plots the silhouette score for each cluster and their average score.
    It measures how similar an object is to its own cluster compared to other clusters.
    """ 
    print("""Cluster silhouette scores: \n 
        +1 = Good 
         0 = close to decision boundary 
        -1 = could be wrongfully assigned \n""")

    K = len(data[cluster].unique())
    cluster_assignment = data[cluster]
    
    # Create a subplot with 1 row and 2 columns
    fig, (ax1) = plt.subplots(1, 1)
    fig.set_size_inches(18, 7)
    ax1.set_xlim([-1, 1])
    ax1.set_ylim([0, len(data) + (K + 1) * 10])   # The (k+1)*10 is for inserting blank space between silhouette plots of individual clusters, to demarcate them clearly.
    
    # Average Silhouette score for all samples.
    # This gives a perspective into the density and separation of the formed clusters
    silhouette_avg = silhouette_score(data[columns_gp], cluster_assignment) # Compute
    ax1.axvline(x=silhouette_avg, color="red", linestyle="--")  # Plot the vertical line for average silhouette score of all the values
    print(f' Average: {silhouette_avg}')
    
    # Compute the silhouette scores for each sample
    sample_silhouette_values = silhouette_samples(data[columns_gp], cluster_assignment)
    print(sample_silhouette_values.shape)

    y_lower = 10
    for i in np.sort(cluster_assignment.unique()):
        # Aggregate the silhouette scores for samples belonging to cluster i, and sort them
        silhouette_values_i = sample_silhouette_values[cluster_assignment == i] # Subset
        silhouette_values_i.sort() #Sort values

        # Calculate average silhouette score of the cluster
        silhouette_avg_i = silhouette_values_i.mean()
        print(f' Average cluster {i}: {silhouette_avg_i}')

        # Calculate cluster size
        size_cluster_i = silhouette_values_i.shape[0]
        y_upper = y_lower + size_cluster_i

        # Format Colors
        spectral_palette = sns.color_palette("Spectral", K)
        color = spectral_palette #[spectral_palette[i] for i in desired_order_colors][i]
        
        ax1.fill_betweenx(
            np.arange(y_lower, y_upper),
            0,
            silhouette_values_i,
            facecolor=color,
            edgecolor=color,
            alpha=0.7,
        )
    
        # Label the silhouette plots with their cluster numbers at the middle
        ax1.text(-0.1, y_lower + 0.5 * size_cluster_i, f'{i} ({size_cluster_i})')
    
        # Compute the new y_lower for next plot
        y_lower = y_upper + 10  # 10 for the 0 samples

    # Format Graph
    ax1.set_title(f'The average silhouette_score is : {silhouette_avg}')
    ax1.set_xlabel("The silhouette coefficient values")
    ax1.set_ylabel("Cluster label")
    ax1.set_yticks([])  # Clear the yaxis labels / ticks
    ax1.set_xticks(np.arange(-1, 1.2, 0.2))
    
    plt.suptitle(f"Silhouette analysis for k = {cluster}", fontsize=14, fontweight="bold")
    plt.savefig(os.path.join(figure_storage, f'PsAID_MixtureModels_Silhouette_{cluster}.png'))
    # plt.show()

In [600]:
if compute_silhouette:
    for cluster in cluster_columns[:4]:
        compute_silhouette_plot(data_complete[columns_gp + [cluster]], columns_gp, cluster, figure_storage='/Users/pattydegroot/Documents/Python/PsAID-Cluster-Analysis/PsAID_Mixture Models/Figures/Silhouettes')

### Pain Visualisation

### Pairplot Heatmap

In [601]:
def pairplot_heatmap(data, varlist, describe_clusters, dictionary, custom_color_map, storefigures, figure_storage): 
    """
    Function that creates a plair plot of the various variables against each other.
    The diagonal shows the KDE
    The colors indicate the assigned clusters
    The hues indicate the density of datapoints in a certain place 

    Before running this function make sure that your clusters have the desired_layer_order
    """
    clusters = np.sort(data[describe_clusters].unique())

    # Order clusters by WCSS for plotting
    cluster_sizes = data[describe_clusters].value_counts()
    cluster_variances = (data.groupby(describe_clusters)[varlist].var().sum(axis=1))  # total variance per cluster
    layer_order = cluster_variances.sort_values().index.tolist() # most compact drawn on top
    
    # Create Pairplot
    pp = sns.pairplot(data[varlist + [describe_clusters]], hue=describe_clusters, hue_order = layer_order[::-1], corner=True, diag_kind='kde', palette=custom_color_map)
    pp.map_lower(sns.kdeplot, levels = 8, shade=True, alpha=0.6, legend = False) #Add filled contours
   # pp.fig.set_size_inches(24, 20)

    # Format axes
    for i, j in zip(*np.tril_indices_from(pp.axes, k=-1)):  # Loop only through the lower triangle
         if pp.axes[i, j] is not None:
            x_label = '\n'.join(textwrap.wrap(dictionary[varlist[j]], width=20))
            y_label = '\n'.join(textwrap.wrap(dictionary[varlist[i]], width=20))
            pp.axes[i, j].set_xlabel(x_label, labelpad=10, fontsize=15)
            pp.axes[i, j].set_ylabel(y_label, labelpad=10, fontsize=15)
            pp.axes[i, j].set(xlim=(0,10), ylim = (0,10))

    # Add legend
    legend_handles = []
    for label, color in custom_color_map.items():
        size = cluster_sizes.get(label, 0)  # default 0 if missing
        legend_label = f"{label} (n={size})"
        patch = mpatches.Patch(color=color, label=legend_label)
        legend_handles.append(patch)
    pp._legend.remove()  # Remove Seaborn’s auto-legend
    
    pp.fig.legend(
        handles=legend_handles,
        title="Clusters",
        loc="upper right",
        bbox_to_anchor=(0.98, 0.98),
        frameon=True,
        fontsize=15
        )

    # Print & store figure
    plt.tight_layout()
    pp.fig.suptitle(f'PsAID clustering, {describe_clusters}', y=1.02, fontsize=35)
    if storefigures:
        pp.savefig(os.path.join(figure_storage, f'PsAID_{describe_clusters}_KDEHeatmap.png'))
    plt.show() 

In [602]:
## Color Settings
if compute_pairplot:
    clusters = sorted(data_complete[describe_cluster].unique())
    custom_palette = sns.color_palette("colorblind", len(clusters))
    custom_color_map = {label: color for label, color in zip(clusters, custom_palette)}
    

    pairplot_heatmap(data_complete[columns_gp+[describe_cluster]], columns_gp, describe_cluster, domains_dict, custom_color_map, storefigures=None, figure_storage=None)

## Keep clustering solutions

In [603]:
# drop_configs = [  'clusterID_gsemWeibull_k3',
#                   'clusterID_gsemWeibull_k4',
#                   'clusterID_gsemWeibull_k5',
#                   'clusterID_gsemWeibull_k6',
#                   'clusterID_gsemWeibull_k7']

In [604]:
# data_complete = data_complete.drop(columns=drop_configs)
# data_complete.info()

In [605]:
## Save Dataset
folder_path = "/Users/pattydegroot/Documents/Python/PsAID-Cluster-Analysis/PsAID_Mixture Models"
file_name = f'PsAID_Mixture Models_Clusters'

# Ensure the folder exists
os.makedirs(folder_path, exist_ok=True)

# Save
data_complete.to_csv(f'{folder_path}/{file_name}.csv', index=False)
print(f' Dataset saved to {folder_path}/{file_name}.csv')

 Dataset saved to /Users/pattydegroot/Documents/Python/PsAID-Cluster-Analysis/PsAID_Mixture Models/PsAID_Mixture Models_Clusters.csv


# Robustness

In [606]:
describe_cluster = 'clusterID_gsemGauss_k6'

In [607]:
# Establish parity between the assigned clusters generated with the full and split data.
def match_clusters(data1, cluster_name1, data2, cluster_name2, k, byvar):
    """
    Function that pairs the cluster centers by closeness. 
    It is used to establish the same cluster codes between algorithms.
    """

    print('Match clusters')

    data1 = data1.copy()
    data1[cluster_name1] = data1[cluster_name1]-1
    data2 = data2.copy()
    data2[cluster_name2] = data2[cluster_name2]-1

    print(f'data1: {data1[cluster_name1].unique()}')
    print(f'data2: {data2[cluster_name2].unique()}')

    centers1 = data1[byvar + [cluster_name1]].groupby(cluster_name1).mean().values
    centers2 = data2[byvar + [cluster_name2]].groupby(cluster_name2).mean().values
    
    # Find Closest Clusters
    distances = cdist(centers1, centers2, metric='euclidean') #Compute pairwise distances
    print("Pairwise Euclidean distances between cluster centers:\n", distances)
    # print("Each row in the distances matrix corresponds to a cluster from data1. Each column in the distances matrix corresponds to a cluster from data2. \n")

    row_ind, col_ind = linear_sum_assignment(distances) #Match clusters using the Hungarian algorithm
    cluster_mapping = dict(zip(col_ind, row_ind)) # Create a mapping from data1 to data2
    
    print(f'Cluster mapping (data2 -> data1): {cluster_mapping} \n')
 
    ## Remap Cluster Assignments
    def remap_labels(labels, mapping):
        """
        Remap cluster labels based on a mapping dictionary.
        :param labels: Original cluster labels.
        :param mapping: Mapping dictionary (e.g., {0: 1, 1: 0, ...}).
        :return: Remapped cluster labels.
        """
        return np.array([mapping[label] for label in labels])

    data2[cluster_name2+'_OG'] = data2[cluster_name2].astype('category')
    data2[cluster_name2] = remap_labels(data2[cluster_name2], cluster_mapping) #Remap cluser assignments
    data2[cluster_name2] = data2[cluster_name2].astype('category') # Convert from int to category

    ## Remap Cluster Centers
    cluster_centers = {
        f'data1': [data1.groupby(cluster_name1)[columns_gp].mean().values], #Compute cluster means
        f'data2': [data2.groupby(cluster_name2)[columns_gp].mean().values]  #Compute cluster means
    }  
    
    # Check if correct
    table = pd.DataFrame({
        f'data1': pd.Series(data1[cluster_name1]).value_counts(),
        f'Original {data2} data2': pd.Series(data2[cluster_name2+'_OG']).value_counts(),
        f'Remapped {data2} data2': pd.Series(data2[cluster_name2]).value_counts()
    }).sort_index()
    
    # print(table)

    # Define output
    return data2

# Compute similarity with the Adjusted Rand Index
def compute_ARI(ARI_stored, estimator1, estimator2, describe_cluster, suffix):
    """
    The Adjusted Rand Index (ARI): a measure used to evaluate the similarity between two data clusterings. 
    It is an adjustment of the Rand Index (RI), which counts the number of pairs of elements that are either both in the same cluster or both in different clusters in the predicted and true clusterings. 
    The ARI adjusts for the chance grouping of elements, providing a more accurate measure of clustering similarity. 
    The ARI adjusts the RI by considering the expected similarity of all pairwise comparisons based on chance. 
    This adjustment makes the ARI more reliable and interpretable, especially when comparing different clustering results.
    
    The ARI score ranges from -1 to 1.
    1 = identical clustering 0 = similarity is no better than what would be expected by chance -1 = two clusters are less similar than random clustering (strongly disagreeing)
    """
    ari = adjusted_rand_score(estimator1, estimator2)
    print(f'Adjusted Rand Index: {ari} \n')

    if suffix == '_split1':
        ari_computed = '_split_sample'
    elif re.match(r'_no_gp\d{2}$', suffix):
        ari_computed = '_variable_validation'
    
    ARI_dict = {
        'cluster_name': describe_cluster,
        f'ARI{ari_computed}': ari,
    }
    
    ARI_stored.append(ARI_dict)
    
    return ARI_stored

## Split Sample Validation

In [608]:
# Import GSEM Cluster Data
splitsample_Mixturedata = pd.read_stata(f'/Users/pattydegroot/Documents/Python/PsAID-Cluster-Analysis/PsAID_Mixture Models/PsAID_GSEM_clustering_splitsample.dta') # Stata data geformat T IN 'MixtureModels.ipynb'

# Rename cluster vars for compatibility
def format_variable(name):
    name = re.sub(r'^cluster_', 'clusterID_', name, flags=re.IGNORECASE) # Replace "Cluster_" (case-insensitive) with "clusterID_
    name = re.sub(r'_(K\d+)$', lambda m: '_' + m.group(1).lower(), name) # Lowercase the final part if it matches 'K' followed by digits
    return name
    
splitsample_cluster_columns = [i for i in splitsample_Mixturedata.columns if re.match(r'^Cluster',i)]
rename_map = {col: format_variable(col) for col in splitsample_cluster_columns} # Build the rename mapping
splitsample_Mixturedata.rename(columns=rename_map, inplace=True) # Rename in the DataFrame
splitsample_cluster_columns = [i for i in splitsample_Mixturedata.columns if re.match(r'^clusterID_',i)]

## Format MM
splitsample_Mixturedata['mm'] = splitsample_Mixturedata['mm'].replace(dict(zip(mm_categoriesA ,mm_categoriesB))) #replace original with new categories
splitsample_Mixturedata[['mm']+splitsample_cluster_columns] = splitsample_Mixturedata[['mm'] + splitsample_cluster_columns].astype('int')# Convert objects to integers
assert splitsample_Mixturedata['mm'].dtype == 'int' #Check if the mm are indeed integers

# Check if converted
unique_values_mm = sorted(splitsample_Mixturedata['mm'].unique())
print(unique_values_mm) #check if replaced
splitsample_Mixturedata['mm'].info()

# Subset the GSEM data
splitsample_Mixturedata = splitsample_Mixturedata[['respondentid', 'mm'] + splitsample_cluster_columns] 
splitsample_Mixturedata = splitsample_Mixturedata.rename(columns={col: f"{col}_split1" for col in cluster_columns}) # Add suffix
splitsample_cluster_columns = [i for i in splitsample_Mixturedata.columns if i.endswith('_split1')] # Update list
print(splitsample_cluster_columns)
splitsample_Mixturedata.head() # Check data

# Merge GSEM data with data_complete[GP01]
print(f'Shape_prior: {data_complete.shape}')
data_split_sample = pd.merge(data_complete, splitsample_Mixturedata, on = ['respondentid', 'mm'], how = 'left')
print(f'Shape_after: {data_split_sample.shape}')
data_split_sample.columns

# Drop all rows where _split1 has no cluster assignment. Meaning it was not included int he cluster computation.
data_split_sample = data_split_sample.dropna(subset=['clusterID_gsemGauss_k3_split1'])
print(f'Shape_after: {data_split_sample.shape}')
print(f'Num NAN: {data_split_sample ['clusterID_gsemGauss_k3_split1'].isna().sum()}') 

data_split_sample.info()

[0, 3, 6, 9, 12, 18, 24, 36, 48, 60, 72, 84, 96]
<class 'pandas.core.series.Series'>
RangeIndex: 2550 entries, 0 to 2549
Series name: mm
Non-Null Count  Dtype
--------------  -----
2550 non-null   int64
dtypes: int64(1)
memory usage: 20.1 KB
['clusterID_gsemGauss_k3_split1', 'clusterID_gsemGauss_k4_split1', 'clusterID_gsemGauss_k5_split1', 'clusterID_gsemGauss_k6_split1', 'clusterID_gsemGauss_k7_split1', 'clusterID_gsemPoisson_k3_split1', 'clusterID_gsemPoisson_k4_split1', 'clusterID_gsemPoisson_k5_split1', 'clusterID_gsemPoisson_k6_split1', 'clusterID_gsemPoisson_k7_split1', 'clusterID_gsemWeibull_k3_split1', 'clusterID_gsemWeibull_k4_split1', 'clusterID_gsemWeibull_k5_split1', 'clusterID_gsemWeibull_k6_split1', 'clusterID_gsemWeibull_k7_split1']
Shape_prior: (5099, 52)
Shape_after: (5099, 67)
Shape_after: (2550, 67)
Num NAN: 0
<class 'pandas.core.frame.DataFrame'>
Index: 2550 entries, 0 to 5097
Data columns (total 67 columns):
 #   Column                           Non-Null Count  Dty

In [609]:
def split_sample_validation(ARI_split_sample, data, describe_cluster, suffix, varlist):
    k = int(re.search(r'k(\d+)', describe_cluster).group(1))
    
    # Match clusters computed with and without varB
    data_varlistB_matched = match_clusters(data, describe_cluster, data, describe_cluster+suffix, k, varlist)
    
    # ARI
    ari_split_sample = compute_ARI(ARI_split_sample, data_varlistB_matched[describe_cluster+suffix], data[describe_cluster], describe_cluster+suffix, suffix)
    
    return ari_split_sample

In [610]:
# if computeARIs == True:
ARI_split_sample = []
    
for describe_cluster in cluster_columns:
        with contextlib.redirect_stdout(io.StringIO()): #Suppress prints inside the function.
            ari_split_sample  = split_sample_validation(ARI_split_sample, data_split_sample, describe_cluster, '_split1', columns_gp)
        print(f'computed ARI {describe_cluster}')
        
pd.DataFrame(ari_split_sample)

computed ARI clusterID_gsemGauss_k3
computed ARI clusterID_gsemGauss_k4
computed ARI clusterID_gsemGauss_k5
computed ARI clusterID_gsemGauss_k6
computed ARI clusterID_gsemGauss_k7
computed ARI clusterID_gsemPoisson_k3
computed ARI clusterID_gsemPoisson_k4
computed ARI clusterID_gsemPoisson_k5
computed ARI clusterID_gsemPoisson_k6
computed ARI clusterID_gsemPoisson_k7
computed ARI clusterID_gsemWeibull_k3
computed ARI clusterID_gsemWeibull_k4
computed ARI clusterID_gsemWeibull_k5
computed ARI clusterID_gsemWeibull_k6
computed ARI clusterID_gsemWeibull_k7


,cluster_name,ARI_split_sample
0,clusterID_gsemGauss_k3_split1,0.980203
1,clusterID_gsemGauss_k4_split1,0.987087
2,clusterID_gsemGauss_k5_split1,0.803905
3,clusterID_gsemGauss_k6_split1,0.925616
4,clusterID_gsemGauss_k7_split1,0.959865
5,clusterID_gsemPoisson_k3_split1,0.934456
6,clusterID_gsemPoisson_k4_split1,0.963389
7,clusterID_gsemPoisson_k5_split1,0.876527
8,clusterID_gsemPoisson_k6_split1,0.905970
9,clusterID_gsemPoisson_k7_split1,0.568247


## Leave Pain Out Validation

In [611]:
# Import GSEM Cluster Data
noGP01_Mixturedata = pd.read_stata(f'/Users/pattydegroot/Documents/Python/PsAID-Cluster-Analysis/PsAID_Mixture Models/PsAID_GSEM_clustering_noGP01.dta') # Stata data geformat T IN 'MixtureModels.ipynb'

# Rename cluster vars for compatibility
def format_variable(name):
    name = re.sub(r'^cluster_', 'clusterID_', name, flags=re.IGNORECASE) # Replace "Cluster_" (case-insensitive) with "clusterID_
    name = re.sub(r'_(K\d+)$', lambda m: '_' + m.group(1).lower(), name) # Lowercase the final part if it matches 'K' followed by digits
    return name
    
noGP01_cluster_columns = [i for i in noGP01_Mixturedata.columns if re.match(r'^Cluster',i)]
rename_map = {col: format_variable(col) for col in noGP01_cluster_columns} # Build the rename mapping
noGP01_Mixturedata.rename(columns=rename_map, inplace=True) # Rename in the DataFrame
noGP01_cluster_columns = [i for i in noGP01_Mixturedata.columns if re.match(r'^clusterID_',i)]

## Format MM
noGP01_Mixturedata['mm'] = noGP01_Mixturedata['mm'].replace(dict(zip(mm_categoriesA ,mm_categoriesB))) #replace original with new categories
noGP01_Mixturedata[['mm']+noGP01_cluster_columns] = noGP01_Mixturedata[['mm'] + noGP01_cluster_columns].astype('int')# Convert objects to integers
assert noGP01_Mixturedata['mm'].dtype == 'int' #Check if the mm are indeed integers

# Check if converted
unique_values_mm = sorted(noGP01_Mixturedata['mm'].unique())
print(unique_values_mm) #check if replaced
noGP01_Mixturedata['mm'].info()

# Subset the GSEM data
noGP01_columns_gp = list(set(columns_gp) - {'gp01'})
noGP01_Mixturedata = noGP01_Mixturedata[['respondentid', 'mm'] + noGP01_cluster_columns] 
noGP01_Mixturedata = noGP01_Mixturedata.rename(columns={col: f"{col}_no_gp01" for col in cluster_columns})
noGP01_Mixturedata.head() # Check data

# Merge GSEM data with data_complete[GP01]
print(f'Shape_prior: {data_complete.shape}')
data_complete = pd.merge(data_complete, noGP01_Mixturedata, on = ['respondentid', 'mm'], how = 'left')
print(f'Shape_after: {data_complete.shape}')
noGP01_Mixturedata.info()


[0, 3, 6, 9, 12, 18, 24, 36, 48, 60, 72, 84, 96, 108]
<class 'pandas.core.series.Series'>
RangeIndex: 5099 entries, 0 to 5098
Series name: mm
Non-Null Count  Dtype
--------------  -----
5099 non-null   int64
dtypes: int64(1)
memory usage: 40.0 KB
Shape_prior: (5099, 52)
Shape_after: (5099, 67)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5099 entries, 0 to 5098
Data columns (total 17 columns):
 #   Column                            Non-Null Count  Dtype  
---  ------                            --------------  -----  
 0   respondentid                      5099 non-null   float64
 1   mm                                5099 non-null   int64  
 2   clusterID_gsemGauss_k3_no_gp01    5099 non-null   int64  
 3   clusterID_gsemGauss_k4_no_gp01    5099 non-null   int64  
 4   clusterID_gsemGauss_k5_no_gp01    5099 non-null   int64  
 5   clusterID_gsemGauss_k6_no_gp01    5099 non-null   int64  
 6   clusterID_gsemGauss_k7_no_gp01    5099 non-null   int64  
 7   clusterID_gsemPoisson_k3_n

In [612]:
def variable_validation(ARI_LeaveOneOut, data, describe_cluster, suffix, varlist):
    k = int(re.search(r'k(\d+)', describe_cluster).group(1))
    
    # Match clusters computed with and without varB
    data_varlistB_matched = match_clusters(data, describe_cluster, data, describe_cluster+suffix, k, varlist)
    
    # ARI
    ari_LeaveOneOut = compute_ARI(ARI_LeaveOneOut, data_varlistB_matched[describe_cluster+suffix], data[describe_cluster], describe_cluster+suffix, suffix)
    
    return ari_LeaveOneOut 

In [613]:
ARI_LeaveOneOut = []
    
for describe_cluster in cluster_columns:
    #for var in columns_gp:
        varlistB = noGP01_columns_gp  #[i for i in psaid12.columns if re.match(r'^gp\d{2}$', i) and i != var]
        with contextlib.redirect_stdout(io.StringIO()): #Suppress prints inside the function.
            ari_LeaveOneOut  = variable_validation(ARI_LeaveOneOut, data_complete, describe_cluster, f'_no_gp01', varlistB)
        print(f'computed ARI {describe_cluster}, without gp01 ({domains_dict['gp01']})')
        
ari_LeaveOneOut = pd.DataFrame(ari_LeaveOneOut)
ari_LeaveOneOut    

computed ARI clusterID_gsemGauss_k3, without gp01 (Pain)
computed ARI clusterID_gsemGauss_k4, without gp01 (Pain)
computed ARI clusterID_gsemGauss_k5, without gp01 (Pain)
computed ARI clusterID_gsemGauss_k6, without gp01 (Pain)
computed ARI clusterID_gsemGauss_k7, without gp01 (Pain)
computed ARI clusterID_gsemPoisson_k3, without gp01 (Pain)
computed ARI clusterID_gsemPoisson_k4, without gp01 (Pain)
computed ARI clusterID_gsemPoisson_k5, without gp01 (Pain)
computed ARI clusterID_gsemPoisson_k6, without gp01 (Pain)
computed ARI clusterID_gsemPoisson_k7, without gp01 (Pain)
computed ARI clusterID_gsemWeibull_k3, without gp01 (Pain)
computed ARI clusterID_gsemWeibull_k4, without gp01 (Pain)
computed ARI clusterID_gsemWeibull_k5, without gp01 (Pain)
computed ARI clusterID_gsemWeibull_k6, without gp01 (Pain)
computed ARI clusterID_gsemWeibull_k7, without gp01 (Pain)


,cluster_name,ARI_variable_validation
0,clusterID_gsemGauss_k3_no_gp01,0.902845
1,clusterID_gsemGauss_k4_no_gp01,0.917395
2,clusterID_gsemGauss_k5_no_gp01,0.832316
3,clusterID_gsemGauss_k6_no_gp01,0.908685
4,clusterID_gsemGauss_k7_no_gp01,0.778694
5,clusterID_gsemPoisson_k3_no_gp01,0.934577
6,clusterID_gsemPoisson_k4_no_gp01,0.865574
7,clusterID_gsemPoisson_k5_no_gp01,0.860767
8,clusterID_gsemPoisson_k6_no_gp01,0.839400
9,clusterID_gsemPoisson_k7_no_gp01,0.824019


## Leave One Out for Gauss_K6

In [614]:
## For K6 only
ARI_LeaveOneOut2 = []
parent_folder = "/Users/pattydegroot/Documents/Python/PsAID-Cluster-Analysis/PsAID_Mixture Models/Leave One Out Analysis"
describe_cluster = 'clusterID_gsemGauss_k6'
data_complete = data_complete.rename(columns={'clusterID_gsemGauss_k6_no_gp01': 'clusterID_gsemGauss_k6_no_gp01_A'})


for folder in os.listdir(parent_folder):
    if not folder.endswith('.dta'):
        continue
    
    print(f'\n {folder} \n')
    var_out = re.search(r'GP\d{2}', folder).group()
    
    ## IMPORT DATASET
    no_GP = pd.read_stata(f'{parent_folder}/{folder}')[['respondentid', 'mm', f'no{var_out}_clusters']] # Stata data geformat T IN 'MixtureModels.ipynb'
    
    # Rename cluster vars for compatibility
    no_GP = no_GP.rename(columns={f'no{var_out}_clusters': 'clusterID_gsemGauss_k6'})
    no_GP_cluster_columns = ['clusterID_gsemGauss_k6']

    ## Format MM
    no_GP['mm'] = no_GP['mm'].replace(dict(zip(mm_categoriesA ,mm_categoriesB))) #replace original with new categories
    no_GP[['mm']+no_GP_cluster_columns] = no_GP[['mm'] + no_GP_cluster_columns].astype('int')# Convert objects to integers
    assert no_GP['mm'].dtype == 'int' #Check if the mm are indeed integers
    
    # Check if converted
    unique_values_mm = sorted(no_GP['mm'].unique())
   # print(unique_values_mm) #check if replaced
    no_GP['mm'].info()

    # Add suffix
    no_GP = no_GP.rename(columns={col: f"{col}_no_{var_out.lower()}" for col in cluster_columns})

    # Merge GSEM data with data_complete[GP01]
    print(f'Shape_prior: {data_complete.shape}')
    data_complete = pd.merge(data_complete, no_GP, on = ['respondentid', 'mm'], how = 'left')
    print(f'Shape_after: {data_complete.shape}')
    no_GP.info()

    ## COMPUTE ARI
    varlistB = new_lst = [x for x in columns_gp if x != f'{var_out.lower()}']
    with contextlib.redirect_stdout(io.StringIO()): #Suppress prints inside the function.
            ari_LeaveOneOut2  = variable_validation(ARI_LeaveOneOut2, data_complete, describe_cluster, f'_no_{var_out.lower()}', varlistB)
    print(f'computed ARI {describe_cluster}, without {var_out} ({domains_dict[var_out.lower()]})')
        
ari_LeaveOneOut2 = pd.DataFrame(ari_LeaveOneOut2)
ari_LeaveOneOut2.sort_values('cluster_name')    


 gsemGaussK6_noGP08.dta 

<class 'pandas.core.series.Series'>
RangeIndex: 5099 entries, 0 to 5098
Series name: mm
Non-Null Count  Dtype
--------------  -----
5099 non-null   int64
dtypes: int64(1)
memory usage: 40.0 KB
Shape_prior: (5099, 67)
Shape_after: (5099, 68)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5099 entries, 0 to 5098
Data columns (total 3 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   respondentid                    5099 non-null   float64
 1   mm                              5099 non-null   int64  
 2   clusterID_gsemGauss_k6_no_gp08  5099 non-null   int64  
dtypes: float64(1), int64(2)
memory usage: 119.6 KB
computed ARI clusterID_gsemGauss_k6, without GP08 (Coping)

 gsemGaussK6_noGP09.dta 

<class 'pandas.core.series.Series'>
RangeIndex: 5099 entries, 0 to 5098
Series name: mm
Non-Null Count  Dtype
--------------  -----
5099 non-null   int64
dtypes: int64(1)
memory usage

,cluster_name,ARI_variable_validation
4,clusterID_gsemGauss_k6_no_gp01,0.908685
2,clusterID_gsemGauss_k6_no_gp02,0.913333
3,clusterID_gsemGauss_k6_no_gp03,0.973656
6,clusterID_gsemGauss_k6_no_gp04,0.778304
7,clusterID_gsemGauss_k6_no_gp05,0.750034
11,clusterID_gsemGauss_k6_no_gp06,0.881426
9,clusterID_gsemGauss_k6_no_gp07,0.953505
0,clusterID_gsemGauss_k6_no_gp08,0.932930
1,clusterID_gsemGauss_k6_no_gp09,0.893285
5,clusterID_gsemGauss_k6_no_gp10,0.935950
